![Git-to-Bronze.jpg](./Git-to-Bronze.jpg "Git-to-Bronze.jpg")

In [0]:
import shutil
import os

# --- 1. CONFIGURACIÓN DE RUTAS ---
base_volume = "/Volumes/ecomotive/landing/ecomotive_vol"
current_user = spark.sql("SELECT current_user()").first()[0]
base_workspace = f"/Workspace/Users/{current_user}/preparacion_databricks_data_engineer/materiales"

# Definición de Orígenes (Workspace - Problemáticos para Spark)
src_drivers = f"{base_workspace}/inbox_drivers"
src_sensors = f"{base_workspace}/inbox_sensors"

# Definición de Destinos (Volumen - Seguros para Spark)
dst_drivers = f"{base_volume}/inbox_drivers"
dst_sensors = f"{base_volume}/inbox_sensors"

# Rutas de Sistema (Checkpoints de Autoloader)
path_sys_drivers = f"{base_volume}/sys/checkpoints/drivers"
path_sys_sensors = f"{base_volume}/sys/checkpoints/sys_sensors"

# Tablas Destino
table_drivers = "ecomotive.bronze.bronze_drivers"
table_sensors = "ecomotive.bronze.bronze_sensors"

print(f"--- CONFIGURACIÓN ---")
print(f"Usuario: {current_user}")
print(f"Origen Drivers: {src_drivers}")
print(f"Origen Sensors: {src_sensors}")

# --- 2. LIMPIEZA DE SISTEMA (RESET) ---
print("\n--- LIMPIEZA DE TABLAS Y CHECKPOINTS ---")
# Borramos tablas
spark.sql(f"DROP TABLE IF EXISTS {table_drivers}")
spark.sql(f"DROP TABLE IF EXISTS {table_sensors}")
print(f"Tablas eliminadas: {table_drivers}, {table_sensors}")

# Borramos checkpoints (Metadata del Autoloader)
dbutils.fs.rm(path_sys_drivers, True)
dbutils.fs.rm(path_sys_sensors, True)
print("Checkpoints eliminados. Autoloader iniciará desde cero.")


--- CONFIGURACIÓN ---
Usuario: pablo.cumbreraconde@gmail.com
Origen Drivers: /Workspace/Users/pablo.cumbreraconde@gmail.com/preparacion_databricks_data_engineer/materiales/inbox_drivers
Origen Sensors: /Workspace/Users/pablo.cumbreraconde@gmail.com/preparacion_databricks_data_engineer/materiales/inbox_sensors

--- LIMPIEZA DE TABLAS Y CHECKPOINTS ---
Tablas eliminadas: ecomotive.bronze.bronze_drivers, ecomotive.bronze.bronze_sensors
Checkpoints eliminados. Autoloader iniciará desde cero.


In [0]:
# --- 3. MIGRACIÓN DE DATOS (COPIA FÍSICA) ---
print("\n--- INICIANDO COPIA DE ARCHIVOS (shutil) ---")

def copy_folder_safely(src, dst, name):
    try:
        if not os.path.exists(src):
            print(f"ADVERTENCIA: La carpeta origen para {name} no existe: {src}")
            return
            
        print(f"Copoyando {name}...")
        print(f"  -> De: {src}")
        print(f"  -> A:  {dst}")
        # dirs_exist_ok=True permite sobreescribir/fusionar si la carpeta ya existía
        shutil.copytree(src, dst, dirs_exist_ok=True) 
        print(f"{name}: Copia exitosa.")
    except Exception as e:
        print(f"{name}: Error crítico al copiar -> {e}")

# Ejecutamos la copia para ambos
copy_folder_safely(src_drivers, dst_drivers, "DRIVERS")
copy_folder_safely(src_sensors, dst_sensors, "SENSORS")

# --- 4. VERIFICACIÓN FINAL ---
print("\n--- CONTENIDO EN VOLUMEN (VERIFICACIÓN) ---")
print("Drivers en Volumen:")
display(dbutils.fs.ls(dst_drivers))

print("Sensors en Volumen:")
display(dbutils.fs.ls(dst_sensors))


--- INICIANDO COPIA DE ARCHIVOS (shutil) ---
Copoyando DRIVERS...
  -> De: /Workspace/Users/pablo.cumbreraconde@gmail.com/preparacion_databricks_data_engineer/materiales/inbox_drivers
  -> A:  /Volumes/ecomotive/landing/ecomotive_vol/inbox_drivers
DRIVERS: Copia exitosa.
Copoyando SENSORS...
  -> De: /Workspace/Users/pablo.cumbreraconde@gmail.com/preparacion_databricks_data_engineer/materiales/inbox_sensors
  -> A:  /Volumes/ecomotive/landing/ecomotive_vol/inbox_sensors
SENSORS: Copia exitosa.

--- CONTENIDO EN VOLUMEN (VERIFICACIÓN) ---
Drivers en Volumen:


path,name,size,modificationTime
dbfs:/Volumes/ecomotive/landing/ecomotive_vol/inbox_drivers/drivers_dirty.csv,drivers_dirty.csv,20104,1766141651000


Sensors en Volumen:


path,name,size,modificationTime
dbfs:/Volumes/ecomotive/landing/ecomotive_vol/inbox_sensors/telemetry_batch_01.json,telemetry_batch_01.json,27458,1766141651000
dbfs:/Volumes/ecomotive/landing/ecomotive_vol/inbox_sensors/telemetry_batch_02.json,telemetry_batch_02.json,48303,1766141651000


In [0]:
df_drivers = (
  spark.readStream
  .format("cloudFiles")
  .option("cloudFiles.format", "csv")
  .option("header", "true")
  .option("cloudFiles.inferColumnTypes", "true")
  .option("cloudFiles.schemaLocation", f"{path_sys_drivers}/schema")
  .option("cloudFiles.schemaEvolutionMode", "rescue")
  .option("cloudFiles.rescuedDataColumn", "_rescued_data")
  .load(dst_drivers)
)

(df_drivers.writeStream
  .format("delta")
  .option("checkpointLocation", f"{path_sys_drivers}/checkpoints")
  .trigger(availableNow=True)
  .table(table_drivers)
  .awaitTermination()
)

print(f"Drivers ingestados en: {table_drivers}")

Drivers ingestados en: ecomotive.bronze.bronze_drivers


In [0]:
# --- INGESTA SENSORS (JSON) ---
df_sensors = (spark.readStream
  .format("cloudFiles")
  .option("cloudFiles.format", "json")
  .option("cloudFiles.inferColumnTypes", "true")
  # Permitimos que entren las columnas nuevas del Batch 02
  .option("cloudFiles.schemaEvolutionMode", "addNewColumns") 
  .option("cloudFiles.schemaLocation", f"{path_sys_sensors}/schema")
  .option("cloudFiles.rescuedDataColumn", "_rescued_data")
  .load(dst_sensors)
)

# Escritura en tabla Delta
(df_sensors.writeStream
  .format("delta")
  .option("checkpointLocation", f"{path_sys_sensors}/checkpoints")
  .option("mergeSchema", "true")
  .trigger(availableNow=True)
  .table(table_sensors)
  .awaitTermination()
)

print(f"Sensores ingestados en: {table_sensors}")

Sensores ingestados en: ecomotive.bronze.bronze_sensors


In [0]:
%sql
SELECT * FROM ecomotive.bronze.bronze_drivers

driver_id,raw_name,license_ref,hourly_wage_text,certifications_list,hired_date,email,_rescued_data
D-101,Miguel Moreno,LIC-816063,22.12 EUR,ElectricVehicle-Hazmat-HeavyLoad,2023-11-25,miguel.moreno@ecomotive.com,null
D-102,CARMEN DIAZ,LIC-697157,37.78,NightShift-ElectricVehicle,2023-06-15,carmen.diaz@ecomotive.com,null
D-103,Elena Navarro,null,23.3,Hazmat-International,2023-11-17,elena.navarro@ecomotive.com,null
D-104,Jorge Alvarez,LIC-273587,34.72 EUR,ElectricVehicle,2023-09-20,jorge.alvarez@ecomotive.com,null
D-105,Elena Romero,LIC-319656,22.0 EUR,International-HeavyLoad-NightShift,2023-09-24,elena.romero@ecomotive.com,null
D-106,Laura Lopez,LIC-768741,47.51,HeavyLoad-Hazmat-NightShift,2023-06-21,laura.lopez@ecomotive.com,null
D-107,isabel ruiz,LIC-335945,30.62,International-Hazmat-HeavyLoad,2023-09-04,isabel.ruiz@ecomotive.com,null
D-108,David Alvarez,LIC-140454,31.12,International-HeavyLoad-ElectricVehicle,2023-04-17,david.alvarez@ecomotive.com,null
D-109,CARLOS HERNANDEZ,LIC-850449,28.91,HeavyLoad,2023-07-26,carlos.hernandez@ecomotive.com,null
D-110,ana diaz,LIC-468978,32.16 EUR,ElectricVehicle,2023-01-15,ana.diaz@ecomotive.com,null


In [0]:
%sql
SELECT * FROM ecomotive.bronze.bronze_sensors

battery_level,cargo_weight_kg,driver_id,engine_data,error_codes,event_id,location,sensor_id,timestamp,vibration_level,_rescued_data
75.69,2682,D-245,"List(2336, WARNING, 96.0)",List(),evt_200000,"List(36.8566, -7.3768)",TRUCK-013,2025-10-01T13:00:00,0.873,null
67.8,6122,D-112,"List(2459, OK, 80.2)",List(),evt_200001,"List(37.3111, -7.3216)",TRUCK-004,2025-10-01T13:02:00,0.395,null
82.92,4754,D-189,"List(1773, OK, 90.5)",List(),evt_200002,"List(40.0859, -3.4985)",TRUCK-012,2025-10-01T13:04:00,0.144,null
78.94,2808,D-298,"List(2772, OK, 89.7)",List(),evt_200003,"List(39.3351, 0.5495)",TRUCK-001,2025-10-01T13:06:00,0.739,null
86.74,6936,D-156,"List(2772, OK, 88.4)",List(),evt_200004,"List(40.7295, 1.5141)",TRUCK-003,2025-10-01T13:08:00,0.355,null
91.87,7063,D-210,"List(2614, WARNING, 83.0)",List(),evt_200005,"List(36.8781, 2.7364)",TRUCK-020,2025-10-01T13:10:00,0.724,null
31.63,3249,D-134,"List(1585, OK, 93.6)","List(203, 500)",evt_200006,"List(36.4348, -8.9718)",TRUCK-014,2025-10-01T13:12:00,0.011,null
41.31,1306,D-156,"List(1845, OK, 85.6)",List(),evt_200007,"List(41.8181, -0.7336)",TRUCK-003,2025-10-01T13:14:00,0.066,null
29.82,4340,D-277,"List(1698, OK, 82.9)",List(),evt_200008,"List(36.5717, 2.2565)",TRUCK-008,2025-10-01T13:16:00,0.385,null
41.7,4245,D-210,"List(1266, OK, 91.0)","List(404, 101)",evt_200009,"List(39.1011, -4.1225)",TRUCK-020,2025-10-01T13:18:00,0.365,null


In [0]:
%sql
SELECT * FROM ecomotive.bronze.bronze_drivers
WHERE `_rescued_data` IS NOT NULL

driver_id,raw_name,license_ref,hourly_wage_text,certifications_list,hired_date,email,_rescued_data


In [0]:
%sql
SELECT * FROM ecomotive.bronze.bronze_sensors
WHERE `_rescued_data` IS NOT NULL

battery_level,cargo_weight_kg,driver_id,engine_data,error_codes,event_id,location,sensor_id,timestamp,vibration_level,_rescued_data


In [0]:
%sql
DESCRIBE HISTORY ecomotive.bronze.bronze_drivers

version,timestamp,userId,userName,operation,operationParameters,job,notebook,clusterId,readVersion,isolationLevel,isBlindAppend,operationMetrics,userMetadata,engineInfo
1,2025-12-18T15:17:32.000Z,74450723977014,pablo.cumbreraconde@gmail.com,STREAMING UPDATE,"Map(outputMode -> Append, queryId -> b3a5cde5-ec3d-4a5e-9b80-8da8d1688200, epochId -> 0, statsOnLoad -> true)",null,List(1416946442892583),1218-144333-pztwqsbp-v2n,0,WriteSerializable,true,"Map(numRemovedFiles -> 0, numOutputRows -> 200, numOutputBytes -> 10285, numAddedFiles -> 1)",null,Databricks-Runtime/17.3.x-aarch64-photon-scala2.13
0,2025-12-18T15:17:27.000Z,74450723977014,pablo.cumbreraconde@gmail.com,CREATE TABLE,"Map(partitionBy -> [], clusterBy -> [], description -> null, isManaged -> true, properties -> {""delta.enableDeletionVectors"":""true"",""delta.writePartitionColumnsToParquet"":""true"",""delta.enableRowTracking"":""true"",""delta.rowTracking.materializedRowCommitVersionColumnName"":""_row-commit-version-col-ed09a739-7703-4353-b9ed-c80d59b5ab22"",""delta.rowTracking.materializedRowIdColumnName"":""_row-id-col-63734452-558d-486b-9578-68668b7ce749""}, statsOnLoad -> false)",null,List(1416946442892583),1218-144333-pztwqsbp-v2n,null,WriteSerializable,true,Map(),null,Databricks-Runtime/17.3.x-aarch64-photon-scala2.13


In [0]:
%sql
DESCRIBE HISTORY ecomotive.bronze.bronze_sensors

version,timestamp,userId,userName,operation,operationParameters,job,notebook,clusterId,readVersion,isolationLevel,isBlindAppend,operationMetrics,userMetadata,engineInfo
1,2025-12-18T15:17:54.000Z,74450723977014,pablo.cumbreraconde@gmail.com,STREAMING UPDATE,"Map(outputMode -> Append, queryId -> 77df4205-f1b4-439c-b3ff-d35b60ecce7d, epochId -> 0, statsOnLoad -> true)",null,List(1416946442892583),1218-144333-pztwqsbp-v2n,0,WriteSerializable,true,"Map(numRemovedFiles -> 0, numOutputRows -> 251, numOutputBytes -> 16009, numAddedFiles -> 1)",null,Databricks-Runtime/17.3.x-aarch64-photon-scala2.13
0,2025-12-18T15:17:48.000Z,74450723977014,pablo.cumbreraconde@gmail.com,CREATE TABLE,"Map(partitionBy -> [], clusterBy -> [], description -> null, isManaged -> true, properties -> {""delta.enableDeletionVectors"":""true"",""delta.writePartitionColumnsToParquet"":""true"",""delta.enableRowTracking"":""true"",""delta.rowTracking.materializedRowCommitVersionColumnName"":""_row-commit-version-col-b9ad9ce1-0547-4536-9c32-5ba5a42bedbf"",""delta.rowTracking.materializedRowIdColumnName"":""_row-id-col-7c9af0e7-b38f-49b8-84ac-e5f3ea06d2bb""}, statsOnLoad -> false)",null,List(1416946442892583),1218-144333-pztwqsbp-v2n,null,WriteSerializable,true,Map(),null,Databricks-Runtime/17.3.x-aarch64-photon-scala2.13


In [0]:
%sql
SELECT 'Drivers' as tipo, count(*) as total FROM ecomotive.bronze.bronze_drivers
UNION ALL
SELECT 'Sensores' as tipo, count(*) as total FROM ecomotive.bronze.bronze_sensors;

tipo,total
Drivers,200
Sensores,251
